## 📎 Code this notebook uses
*(how to run, and which code each step uses: [`README.md`](README.md))*

- **`code/pipeline/compute/`:** `compute_stability_windows.py`
- **`code/pipeline/figures/`:** `make_joint_figures.py` (Figures 1, 2, SI Figure S4), `make_context_map_figure.py` (SI Figure S1), `make_study_overview.py` (SI Figure S2)
- **`code/src/lunar/`:** `config` (every model input), `apollo_helpers` (the stability windows)

# 01 — The data and the model inputs

What goes into the fit: which Apollo sensors are used, over which part of the
record each one is averaged, and the fixed inputs of the thermal model
(Sections 2.1–2.5 of the letter).

Set the run mode in the next cell; "show" reads the stored results.

In [ ]:
# How much to run (README.md explains the three modes):
#   "show"  : run nothing, read the stored results            (minutes)
#   "quick" : run the short steps and the fast form of long ones  (~30 min for all notebooks)
#   "full"  : run everything from scratch, as for the paper    (about 6 h for all notebooks)
MODE = "show"

import pandas as pd
from lunar import notebook as nb
nb.set_mode(MODE)
SITES_ = ("A15", "A17")

## 1. The sensors and their stability windows (Section 2.1, SI Table S2)

Each sensor's equilibrium temperature $T_\mathrm{eq}$ is its mean over a
*stability window*: a stretch of the record where it no longer drifts. The
windows are found sensor by sensor, so they are not aligned in time. Sensors
in the top 80 cm (the borestem zone) are listed but not fitted.

In [ ]:
nb.step("compute/compute_stability_windows.py", minutes=0.2, writes="stability_windows.json")

In [ ]:
win = nb.load("stability_windows.json")
for s in SITES_:
    df = pd.DataFrame(win[s])
    cols = [c for c in ("probe", "sensor", "depth_cm", "used", "method", "window_start", "window_end",
                        "T_eq_K", "n_samples") if c in df]
    print(f"Apollo {s[1:]}: {int(df['used'].sum())} sensors used of {len(df)}")
    display(df[cols])

In [ ]:
chk = nb.Checks("Compared with the paper (Section 2.1):")
for s, n in (("A15", "7"), ("A17", "16")):
    chk.add_text(f"{s}: sensors used (below 80 cm)", str(sum(r["used"] for r in win[s])), n)
chk.show()

## 2. The model inputs (SI Table S1)

Every fixed input lives in one place, `code/src/lunar/config.py`. The albedo
and $K_d$ are not inputs: notebook 03 fits them.

In [ ]:
from lunar.config import SITES, HAYNE, GRID, DT_STEP
from lunar.constants import RHO_SURFACE, RHO_DEEP
rows = [("latitude (deg N)", SITES["A15"]["lat"], SITES["A17"]["lat"]),
        ("emissivity", SITES["A15"]["emissivity"], SITES["A17"]["emissivity"]),
        ("basal heat flux Q_b (mW m^-2)", SITES["A15"]["Q_BASAL"]*1e3, SITES["A17"]["Q_BASAL"]*1e3),
        ("surface conductivity K_s (mW m^-1 K^-1)", HAYNE["K_S"]*1e3, HAYNE["K_S"]*1e3),
        ("compaction depth H (cm)", HAYNE["H"]*100, HAYNE["H"]*100),
        ("radiative coefficient chi", HAYNE["CHI"], HAYNE["CHI"]),
        ("surface / deep density (kg m^-3)", f"{RHO_SURFACE:.0f} / {RHO_DEEP:.0f}", f"{RHO_SURFACE:.0f} / {RHO_DEEP:.0f}")]
display(pd.DataFrame(rows, columns=["input", "Apollo 15", "Apollo 17"]))
print(f"grid: {GRID}   time step: {DT_STEP:.0f} s")

## 3. Figures

`make_joint_figures.py` writes all the letter's figures and SI Figures S3, S4
at once (about 2 min). In "show" mode it runs only if a figure is missing.

In [ ]:
nb.figures("figures/make_joint_figures.py", minutes=2,
           makes=("fig_joint_intro_column", "fig_joint_apollo_timeline_probes", "fig_joint_thermal_profiles",
                  "fig_joint_constraints", "fig_joint_annual_wave", "fig_joint_mcmc",
                  "fig_joint_anchor_process", "fig_joint_amplitude_vs_depth", "fig_joint_robustness"))

**Figure 1.** The modeled regolith column at the two sites.

In [ ]:
nb.show("fig_joint_intro_column")

**Figure 2.** The Apollo records and each sensor's stability window.

In [ ]:
nb.show("fig_joint_apollo_timeline_probes")

**SI Figure S4.** The day–night temperature swing against depth: the
borestem diagnostic behind the 80 cm cut.

In [ ]:
nb.show("fig_joint_amplitude_vs_depth")

**SI Figure S1.** The landing sites. Needs the LOLA map (notebook 00,
section 4).

In [ ]:
nb.figures("figures/make_context_map_figure.py", makes="fig_context_map", minutes=1,
           needs=("data/lola/ldem_16.img",))
nb.show("fig_context_map", width=700)

**SI Figure S2.** The study at a glance. A diagram, compiled with
pdflatex (TeX Live or MacTeX).

In [ ]:
nb.figures("figures/make_study_overview.py", makes="fig_study_overview", minutes=0.5)
nb.show("fig_study_overview")

Next: **02_anchor_method.ipynb** (how the solver works) or straight to
**03_joint_fit.ipynb** (the result).